# Module 4 — Session 2: LoRA Fine-Tuning
## Goal: Fine-tune Llama 3.2 1B on our Swiggy support dataset using LoRA on a free Colab GPU

In [ ]:
# Unsloth is the library that makes LoRA fine-tuning fast on free Colab GPU
# This installation takes 2-3 minutes — please wait for it to complete

!pip install unsloth -q

In [ ]:
# Check if Unsloth installed successfully
import unsloth
print("Unsloth installed successfully!")
print(f"Version: {unsloth.__version__}")

In [ ]:
from unsloth import FastLanguageModel

# Load the base model and tokenizer
# max_seq_length = maximum number of tokens the model can see at once
# load_in_4bit = compress model weights to 4-bit — saves GPU memory on free Colab

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = 512,
    load_in_4bit = True
)

print("Base model loaded successfully!")

In [ ]:
# Attach LoRA adapters to the model
# We are telling Unsloth — which layers to add adapters to and how big those adapters should be

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                      # size of the LoRA adapter — higher = more capacity but slower
    target_modules = ["q_proj", "v_proj"],  # which layers to attach adapters to
    lora_alpha = 16,             # scaling factor for the adapter weights
    lora_dropout = 0,            # dropout for regularization — 0 means disabled
    bias = "none",               # whether to train bias terms — none keeps it simple
)

print("LoRA adapters attached successfully!")

In [ ]:
# Reload base model fresh
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = 512,
    load_in_4bit = True
)

# Attach LoRA adapters with correct layer names as a list
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],  # correct Llama layer names
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)

print("Base model + LoRA adapters ready!")

In [ ]:
# Install groq to regenerate our dataset
!pip install groq -q

In [ ]:
import json
import os
import random
from groq import Groq
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
client = Groq()

# Our 5 seed examples
seed_examples = [
    {
        "instruction": "You are a Swiggy customer support agent. Be empathetic, helpful, and always offer a solution.",
        "input": "It's been over an hour and my order hasn't arrived. I'm very hungry!",
        "output": "I sincerely apologize for this delay! I've tracked your order and see the delivery partner is 8 minutes away due to heavy rain in your area. I've added ₹75 to your Swiggy wallet for this inconvenience. We value your patience!"
    },
    {
        "instruction": "You are a Swiggy customer support agent. Be empathetic, helpful, and always offer a solution.",
        "input": "I ordered a Veg Burger but received a Chicken Burger. I am vegetarian!",
        "output": "I'm truly sorry for this serious mistake! I've raised a complaint with the restaurant and initiated a full refund of ₹249 to your original payment method within 5-7 business days. We take dietary requirements very seriously."
    },
    {
        "instruction": "You are a Swiggy customer support agent. Be empathetic, helpful, and always offer a solution.",
        "input": "My order was cancelled automatically but money was deducted from my account.",
        "output": "I apologize for this inconvenience! Your payment of ₹380 will be refunded within 5-7 business days. I've added ₹50 to your Swiggy wallet to compensate for the trouble. Please feel free to reorder!"
    },
    {
        "instruction": "You are a Swiggy customer support agent. Be empathetic, helpful, and always offer a solution.",
        "input": "My order arrived but the french fries were missing. I paid ₹80 extra for them!",
        "output": "I'm sorry about the missing fries! I've verified your order and refunded ₹80 to your Swiggy wallet immediately. Thank you for flagging this — it helps us improve!"
    },
    {
        "instruction": "You are a Swiggy customer support agent. Be empathetic, helpful, and always offer a solution.",
        "input": "The delivery partner was very rude to me and threw the food bag at my door.",
        "output": "I sincerely apologize for this unacceptable behavior! I've flagged this incident for immediate review. I've added ₹150 to your Swiggy wallet as compensation. You deserve respectful service always!"
    }
]

print(f"Seed examples ready: {len(seed_examples)}")

In [ ]:
# Generate synthetic examples using Groq
categories = ["order_delay", "wrong_item", "refund_request",
              "missing_item", "app_issue", "delivery_behavior"]

def generate_examples_for_category(category, n=4):
    prompt = f"""You are creating training data for a Swiggy customer support AI.

Generate exactly {n} different customer support examples for the category: {category}

Rules:
- instruction must always be: "You are a Swiggy customer support agent. Be empathetic, helpful, and always offer a solution."
- input must be a realistic Indian customer complaint (1-2 sentences)
- output must be an empathetic agent response (3-4 sentences, always offer a solution)
- Use Indian context: mention ₹ amounts, Indian cities like Mumbai/Delhi/Bengaluru/Chennai

Return ONLY a valid JSON array, no explanation, no markdown, no extra text.
Format:
[
  {{"instruction": "...", "input": "...", "output": "..."}},
  {{"instruction": "...", "input": "...", "output": "..."}}
]"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.8
    )
    return json.loads(response.choices[0].message.content)

# Generate and save
all_examples = seed_examples.copy()
for category in categories:
    print(f"Generating: {category}...")
    try:
        new_examples = generate_examples_for_category(category, n=4)
        all_examples.extend(new_examples)
    except Exception as e:
        print(f"  Error: {e}")

# Shuffle and split
random.seed(42)
random.shuffle(all_examples)
split_point = int(len(all_examples) * 0.8)
train_data = all_examples[:split_point]

# Save train file
with open("swiggy_train.jsonl", "w") as f:
    for item in train_data:
        f.write(json.dumps(item) + "\n")

print(f"\nTotal examples: {len(all_examples)}")
print(f"Train examples saved: {len(train_data)}")

In [ ]:
# This is the Alpaca prompt template Unsloth expects
# Every example gets converted into this single string format

alpaca_prompt = """Below is an instruction with an input. Write a response.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# Test it with one example
train_data_loaded = []
with open("swiggy_train.jsonl", "r") as f:
    for line in f:
        train_data_loaded.append(json.loads(line.strip()))

# Format first example and print
first = train_data_loaded[0]
formatted = alpaca_prompt.format(first["instruction"], first["input"], first["output"])
print(formatted)

In [ ]:
# datasets is a HuggingFace library for loading and formatting training data
# It is already installed with Unsloth — no need to pip install
from datasets import Dataset

def format_examples(examples):
    """
    Convert all JSONL examples into formatted text strings.

    Input: list of dicts with instruction, input, output
    Output: HuggingFace Dataset object
    """
    formatted_texts = []

    for ex in examples:
        # Format each example using our template
        # EOS_TOKEN tells the model where the response ends
        text = alpaca_prompt.format(
            ex["instruction"],
            ex["input"],
            ex["output"]
        ) + tokenizer.eos_token  # eos_token = end of sequence marker

        formatted_texts.append({"text": text})

    # Convert list of dicts into HuggingFace Dataset
    return Dataset.from_list(formatted_texts)

# Format all training examples
train_dataset = format_examples(train_data_loaded)

print(f"Dataset ready!")
print(f"Total examples: {len(train_dataset)}")
print(f"Columns: {train_dataset.column_names}")

In [ ]:
# SFTTrainer is from the trl library — already installed with Unsloth
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",   # which column to train on
    max_seq_length = 512,
    args = TrainingArguments(
        per_device_train_batch_size = 2,   # examples per step
        num_train_epochs = 3,              # how many times to read the full dataset
        learning_rate = 2e-4,             # how fast to learn
        output_dir = "outputs",           # where to save checkpoints
        logging_steps = 1,               # print loss every step
    ),
)

print("Trainer ready!")

In [ ]:
# This runs the actual fine-tuning
# The model will read our 16 Swiggy examples 3 times (num_train_epochs = 3)
# You will see the loss number printed at every step — watch it go down!

trainer.train()

In [ ]:
# Fix: use save_strategy="no" to skip checkpoint saving during training
# This avoids the pickling error we just saw

from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = 512,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        output_dir = "outputs",
        logging_steps = 1,
        save_strategy = "no",    # skip saving checkpoints during training
    ),
)

trainer.train()

In [ ]:
# Switch model to inference mode — this is required after training
FastLanguageModel.for_inference(model)

# A brand new complaint the model has never seen
test_input = alpaca_prompt.format(
    "You are a Swiggy customer support agent. Be empathetic, helpful, and always offer a solution.",
    "I ordered biryani from a restaurant in Chennai but received cold and stale food. This is unacceptable!",
    ""  # output is empty — we want the model to fill this in
)

# Tokenize the input
inputs = tokenizer(test_input, return_tensors="pt").to("cuda")

# Generate the response
outputs = model.generate(**inputs, max_new_tokens=150)

# Decode back to text
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Print only the response part
print(response.split("### Response:")[-1].strip())

Here are the AWS equivalents for everything we did in Session 2:

Unsloth + LoRA fine-tuning on Colab T4 GPU
→ Amazon SageMaker Training Jobs with GPU instances (ml.g5.2xlarge). You upload your dataset to S3 and SageMaker runs the training job on a managed GPU.

Loading base model from HuggingFace
→ Amazon Bedrock Custom Models or SageMaker JumpStart. Both give you access to foundation models like Llama that you can fine-tune without downloading anything manually.

LoRA adapters (training only 0.90% of weights)
→ Amazon Bedrock Fine-tuning uses the same parameter efficient technique internally. You never see it but it is happening under the hood.

JSONL training dataset
→ Amazon S3 for storage + SageMaker Data Wrangler for formatting and preparing the dataset before training.

SFTTrainer — running the training loop
→ SageMaker Training Jobs. You define epochs, batch size, learning rate exactly like we did — just in a config file instead of Python arguments.

Testing the fine-tuned model
→ SageMaker Endpoints. After training you deploy the model to an endpoint and send test requests via API.